In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('C:/Users/Lenovo/churn-predictor')  # repo root

In [3]:
import pandas as pd
import numpy as np
from src.features import (
    run_pipeline,
    build_preprocessor,
    NUMERICAL_FEATURES,
    MULTI_CAT_FEATURES,
    BINARY_FEATURES,
)

In [4]:
# Load all splits
X_train, X_val, X_test, y_train, y_val, y_test = run_pipeline(
    'C:/Users/Lenovo/churn-predictor/data/raw/telco_churn.csv'
)

print("X_train shape:", X_train.shape)
print("Columns:", X_train.columns.tolist())

Train : (4929, 21) | churn rate: 0.265
Val   : (1057, 21)   | churn rate: 0.266
Test  : (1057, 21)  | churn rate: 0.265
X_train shape: (4929, 21)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'charges_per_month', 'num_services']


In [5]:
preprocessor = build_preprocessor()

# Fit ONLY on training data
X_train_transformed = preprocessor.fit_transform(X_train)

# Apply to val and test — no fitting, just transform
X_val_transformed   = preprocessor.transform(X_val)
X_test_transformed  = preprocessor.transform(X_test)

print("Shape after transform:")
print(f"  Train: {X_train_transformed.shape}")
print(f"  Val:   {X_val_transformed.shape}")
print(f"  Test:  {X_test_transformed.shape}")

Shape after transform:
  Train: (4929, 32)
  Val:   (1057, 32)
  Test:  (1057, 32)


In [6]:
def get_feature_names(preprocessor):
    """Extract output column names from a fitted ColumnTransformer."""
    num_names = NUMERICAL_FEATURES
    
    cat_names = (preprocessor
                 .named_transformers_['cat']
                 .get_feature_names_out(MULTI_CAT_FEATURES)
                 .tolist())
    
    bin_names = BINARY_FEATURES
    
    return num_names + cat_names + bin_names

feature_names = get_feature_names(preprocessor)
print(f"\nTotal output features: {len(feature_names)}")
print("\nAll output columns:")
for i, name in enumerate(feature_names):
    print(f"  {i:2d}. {name}")


Total output features: 32

All output columns:
   0. tenure
   1. MonthlyCharges
   2. TotalCharges
   3. charges_per_month
   4. num_services
   5. MultipleLines_No phone service
   6. MultipleLines_Yes
   7. InternetService_Fiber optic
   8. InternetService_No
   9. OnlineSecurity_No internet service
  10. OnlineSecurity_Yes
  11. OnlineBackup_No internet service
  12. OnlineBackup_Yes
  13. DeviceProtection_No internet service
  14. DeviceProtection_Yes
  15. TechSupport_No internet service
  16. TechSupport_Yes
  17. StreamingTV_No internet service
  18. StreamingTV_Yes
  19. StreamingMovies_No internet service
  20. StreamingMovies_Yes
  21. Contract_One year
  22. Contract_Two year
  23. PaymentMethod_Credit card (automatic)
  24. PaymentMethod_Electronic check
  25. PaymentMethod_Mailed check
  26. gender
  27. Partner
  28. Dependents
  29. PhoneService
  30. PaperlessBilling
  31. SeniorCitizen


In [7]:
# Convert to dataframe for easy inspection
X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)

print("Numerical feature stats after scaling (should be mean≈0, std≈1 on train):")
print(X_train_df[NUMERICAL_FEATURES].describe().round(3))

Numerical feature stats after scaling (should be mean≈0, std≈1 on train):
         tenure  MonthlyCharges  TotalCharges  charges_per_month  num_services
count  4929.000        4929.000      4929.000           4929.000      4929.000
mean     -0.000          -0.000         0.000             -0.000        -0.000
std       1.000           1.000         1.000              1.000         1.000
min      -1.313          -1.531        -1.005             -1.912        -1.102
25%      -0.949          -0.982        -0.834             -1.081        -1.102
50%      -0.140           0.187        -0.391              0.059        -0.026
75%       0.953           0.837         0.679              0.861         0.511
max       1.600           1.785         2.807              1.949         2.124


In [8]:
X_val_df = pd.DataFrame(X_val_transformed, columns=feature_names)

print("\nNumerical feature stats on val (mean≠0 is expected and correct):")
print(X_val_df[NUMERICAL_FEATURES].describe().round(3))


Numerical feature stats on val (mean≠0 is expected and correct):
         tenure  MonthlyCharges  TotalCharges  charges_per_month  num_services
count  1057.000        1057.000      1057.000           1057.000      1057.000
mean     -0.014          -0.009        -0.012              0.006        -0.013
std       0.974           0.993         0.992              0.979         0.992
min      -1.273          -1.513        -0.996             -1.600        -1.102
25%      -0.949          -0.857        -0.822             -1.056        -1.102
50%      -0.140           0.174        -0.413              0.060        -0.026
75%       0.872           0.807         0.578              0.819         0.511
max       1.600           1.780         2.739              1.944         2.124


## Why val mean ≠ 0 after scaling

StandardScaler stores the mean and std from the training set.
When it transforms val/test, it uses those stored train statistics.
Val data has a slightly different distribution, so its scaled mean won't be 0.

This is correct and intentional — it simulates production, where incoming
data is scaled using statistics computed at training time, not at inference time.
If we refitted the scaler on val data, we'd be using future information during training.

In [9]:
# Simulate an unseen category arriving at the API
import warnings

test_row = X_train.iloc[[0]].copy()
test_row['Contract'] = 'Week-to-week'   # category that doesn't exist in training

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = preprocessor.transform(test_row)

# All Contract-related columns should be 0 (unknown → all zeros)
contract_cols = [c for c in feature_names if 'Contract' in c]
print("Contract columns for unseen category (should all be 0):")
result_df = pd.DataFrame(result, columns=feature_names)
print(result_df[contract_cols])

Contract columns for unseen category (should all be 0):
   Contract_One year  Contract_Two year
0                0.0                0.0


In [10]:
print("Binary feature unique values after transform (should only be 0 and 1):")
X_train_df2 = pd.DataFrame(X_train_transformed, columns=feature_names)
for col in BINARY_FEATURES:
    unique_vals = sorted(X_train_df2[col].unique())
    print(f"  {col}: {unique_vals}")

Binary feature unique values after transform (should only be 0 and 1):
  gender: [np.float64(0.0), np.float64(1.0)]
  Partner: [np.float64(0.0), np.float64(1.0)]
  Dependents: [np.float64(0.0), np.float64(1.0)]
  PhoneService: [np.float64(0.0), np.float64(1.0)]
  PaperlessBilling: [np.float64(0.0), np.float64(1.0)]
  SeniorCitizen: [np.float64(0.0), np.float64(1.0)]


In [11]:
for name, arr in [('train', X_train_transformed),
                  ('val',   X_val_transformed),
                  ('test',  X_test_transformed)]:
    nan_count = np.isnan(arr).sum()
    print(f"{name}: {nan_count} NaN values — {'✓ clean' if nan_count == 0 else '✗ PROBLEM'}")

train: 0 NaN values — ✓ clean
val: 0 NaN values — ✓ clean
test: 0 NaN values — ✓ clean


In [12]:
import pickle, os

os.makedirs('models', exist_ok=True)

with open('models/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("Preprocessor saved to models/preprocessor.pkl")

# Verify you can reload it
with open('models/preprocessor.pkl', 'rb') as f:
    preprocessor_reloaded = pickle.load(f)

X_check = preprocessor_reloaded.transform(X_val)
assert X_check.shape == X_val_transformed.shape
print("Reload verified — shapes match")

Preprocessor saved to models/preprocessor.pkl
Reload verified — shapes match


In [13]:
print("✓ Notebook runs clean top to bottom")
print(f"  Last run: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")

✓ Notebook runs clean top to bottom
  Last run: 2026-05-27 10:40
